# Çatı güneş paneli uygunluğu ve toplumsal öncelik: Çiğli + Karşıyaka model çalışması

Bu defter projenin üç modelini 53 mahalle için baştan hesaplar:

1. **Model 1 · Çatı alanı:** OSM ve Microsoft bina ayak izlerinden kullanılabilir çatı alanı *bandı*
2. **Model 2 · Üretim:** PVGIS özgül verimiyle kurulu güç (kWp) ve yıllık üretim (kWh) *bandı*
3. **Model 3 · Öncelik:** teknik / ihtiyaç / erişim eksenleri, sıra tabanlı normalizasyon, AHP benzeri ağırlıklar, duyarlılık analizi

**Bu sürümdeki iki değişiklik (11 Eylül 2026)**

| Değişiklik | Önce | Şimdi | Etkisi |
|---|---|---|---|
| Hane tüketimi | 3.600 kWh/yıl (varsayım) | 1.059 kWh/kişi × 2,8 kişi/hane ≈ **2.965 kWh/yıl** | Yalnızca "hane eşdeğeri" sayıları; skorlar değişmez |
| Eşit değerlerin sırası | Dizideki sıraya göre keyfi ayrılıyordu | Eşit değerler **ortalama sıra** alıyor | Erişim ve ihtiyaç puanları, dolayısıyla sıralama |

Hane tüketiminin kaynakları: İZKA *İzmir Mevcut Durum Analizi* (2023), PDF s.255 ve *2024–2028 İzmir Bölge Planı*, PDF s.89.

**Veri nereden geliyor?** Ağır coğrafi adımlar yerel bilgisayarda çalıştı: Microsoft'un 391 bin binalık ayak izi dosyası, OSM Overpass sorguları, kamu binası eşleştirmesi ve PVGIS sorguları. Sonuçları mahalle başına tek satır olarak aşağıdaki hücreye gömülü. Defter internet ya da Drive bağlantısı istemez: **Çalışma zamanı → Tümünü çalıştır**.

## 0 · Kurulum ve girdi tablosu

| Sütun | Anlamı | Kaynak |
|---|---|---|
| `kademe`, `ortusme` | MS binalarının OSM binasıyla örtüşme oranı; A ≥ 0,65 · B ≥ 0,50 · C altı | MS × OSM çapraz kontrol |
| `osm_m2`, `ms_m2` | 25–8.000 m² arası bina ayak izi alanı toplamı | OSM Overpass, Microsoft Global ML Building Footprints |
| `oz_duz`, `oz_guney`, `oz_kuzey` | 1 kWp için yıllık kWh: düz, 30° güney, 30° kuzey | PVGIS v5.3, %14 sistem kaybı |
| `askida_n` | Askıda fatura başvurusu | İzmir BB açık veri (2020) |
| `su_kisi` | Konut abonesi başına ortalama aylık su tüketimi (gelir vekili) | İzmir BB açık veri (2023) |
| `ses` | Sosyoekonomik seviye skoru, **ilçe düzeyi** | TÜİK SES 2023 |
| `kamu_n`, `kamu_m2` | Mülkiyeti kamu olan tesislerdeki binaların çatı alanı | OSM + mülkiyet sınıflandırması |
| `belirsiz_n`, `belirsiz_m2` | Mülkiyeti belirsiz tesisler (skora girmez) | aynı |
| `ref_skor_ahp` | Yerel modelin önceki AHP skoru (yalnızca doğrulama için) | `cache/bolge_modelleri.csv` |

In [ ]:
import io, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option('display.width', 180)
pd.set_option('display.max_columns', 40)

GIRDI = """ilce;mahalle;kademe;ortusme;km2;nufus;lat;lon;osm_n;ms_n;osm_m2;ms_m2;oz_duz;oz_guney;oz_kuzey;askida_n;su_kisi;ses;kamu_n;kamu_m2;belirsiz_n;belirsiz_m2;ref_skor_ahp
ÇİĞLİ;AHMET EFENDİ;A;0.714;0.2812;1586;38.52166;27.03655;112;105;34900;59150;1393.04;1603.98;916.01;4;17.7032;148.13;0;0.0;0;0.0;0.1704
ÇİĞLİ;AHMET TANER KIŞLALI;A;0.779;0.4434;6527;38.49575;27.08573;381;299;78280;95320;1383.23;1587.12;911.95;16;9.4672;148.13;3;1788.5;0;0.0;0.4123
ÇİĞLİ;ATAŞEHİR;A;0.712;3.3571;28799;38.48561;27.06341;1604;1115;434340;565080;1384.62;1592.67;913.02;59;13.3457;148.13;16;24419.8;14;6112.2;0.7373
ÇİĞLİ;ATATÜRK;B;0.585;1.0718;12792;38.51897;27.06394;177;246;72130;116780;1377.17;1580.16;913.54;54;12.2088;148.13;0;0.0;3;1820.2;0.379
ÇİĞLİ;AYDINLIKEVLER;A;0.66;0.6881;4034;38.48996;27.08202;501;418;98450;131510;1384.2;1592.24;912.7;6;11.2272;148.13;2;2469.6;3;1112.0;0.5327
ÇİĞLİ;BALATÇIK;A;0.668;3.0315;18254;38.5142;27.03692;1430;1042;369970;466800;1393.03;1604.2;915.9;61;11.9298;148.13;12;43871.4;13;14171.9;0.8037
ÇİĞLİ;ÇAĞDAŞ;B;0.618;0.3611;6231;38.50305;27.05972;134;136;43090;55900;1372.2;1567.37;909.14;14;8.6712;148.13;4;1040.1;3;1789.6;0.3637
ÇİĞLİ;CUMHURİYET;C;0.442;9.3583;6914;38.54257;27.07793;258;373;63640;95910;1372.03;1566.51;909.64;51;10.2622;148.13;0;0.0;1;99.2;0.3519
ÇİĞLİ;EGEKENT;A;0.677;0.5035;8296;38.5052;27.06357;126;167;53980;81230;1377.2;1579.86;913.36;20;8.7279;148.13;2;3936.5;4;1261.5;0.3879
ÇİĞLİ;ESENTEPE;B;0.568;0.8451;5903;38.51064;27.05484;164;185;58350;86370;1377.04;1579.86;913.19;28;9.6287;148.13;2;2432.7;3;0.0;0.4092
ÇİĞLİ;EVKA-2;A;0.917;0.5921;4222;38.50001;27.08531;268;288;88060;87990;1384.11;1592.5;912.23;18;9.3688;148.13;6;4160.1;4;2455.1;0.5258
ÇİĞLİ;EVKA-5;B;0.513;0.6374;12945;38.51745;27.05329;176;197;66550;96690;1374.92;1573.32;909.54;52;9.2997;148.13;4;8744.6;9;1163.5;0.484
ÇİĞLİ;EVKA-6;A;0.833;0.1611;2296;38.50427;27.08868;31;36;15620;17910;1374.92;1571.65;912.12;3;8.0734;148.13;1;0.0;0;0.0;0.2446
ÇİĞLİ;GÜZELTEPE;B;0.625;0.5271;6476;38.50069;27.07678;1633;1092;119680;159370;1375.73;1574.37;911.27;53;10.3539;148.13;0;0.0;0;0.0;0.5969
ÇİĞLİ;HARMANDALI GAZİ MUSTAFA KEMAL ATATÜRK;A;0.68;4.023;2863;38.53205;27.08107;882;619;92770;116300;1372.87;1574.51;903.61;13;12.0355;148.13;0;0.0;0;0.0;0.44
ÇİĞLİ;İNÖNÜ;A;0.699;5.2524;7905;38.52554;27.08022;647;489;100430;137010;1375.07;1568.1;912.79;56;12.1371;148.13;0;0.0;0;0.0;0.5273
ÇİĞLİ;İZKENT;C;0.394;0.6591;6408;38.51204;27.06646;69;132;30880;60590;1373.02;1569.25;908.62;28;8.8416;148.13;0;0.0;2;0.0;0.3304
ÇİĞLİ;KAKLIÇ;B;0.517;43.4768;1639;38.48677;26.99604;1336;1588;390150;512240;1404.45;1620.35;921.11;2;15.4206;148.13;0;0.0;0;0.0;0.6092
ÇİĞLİ;KÖYİÇİ;A;0.73;0.4625;3988;38.49763;27.06528;1089;574;95610;121740;1383.97;1589.26;911.02;15;9.3799;148.13;1;0.0;5;4130.0;0.5033
ÇİĞLİ;KÜÇÜK ÇİĞLİ;B;0.634;1.0381;21591;38.50163;27.05201;1251;721;329630;419310;1377.15;1579.84;913.6;112;9.825;148.13;2;1029.9;1;0.0;0.7404
ÇİĞLİ;MALTEPE;A;0.692;0.4497;4078;38.49126;27.07458;819;380;103290;125730;1384.6;1592.8;912.36;9;10.569;148.13;5;4301.0;6;2233.5;0.5773
ÇİĞLİ;SASALI MERKEZ;B;0.537;40.9626;4210;38.49323;26.932;1549;2197;407040;604750;1395.41;1613.15;908.66;1;20.6925;148.13;2;3327.4;1;1248.2;0.6954
ÇİĞLİ;ŞİRİNTEPE;B;0.644;0.5955;8512;38.4949;27.07155;1923;1032;164560;214030;1384.37;1591.35;911.07;37;9.6138;148.13;1;638.4;0;0.0;0.6556
ÇİĞLİ;UĞUR MUMCU;B;0.613;0.8589;5894;38.52257;27.04752;556;364;90690;128610;1390.68;1594.87;914.75;27;11.2016;148.13;2;7784.3;0;0.0;0.6244
ÇİĞLİ;YAKAKENT;B;0.514;4.9755;10244;38.51265;27.08536;539;395;76080;122050;1359.87;1552.68;892.38;32;14.4558;148.13;0;0.0;0;0.0;0.359
ÇİĞLİ;YENİ MAHALLE;A;0.684;0.7332;12553;38.49631;27.05253;1209;541;187630;248780;1384.61;1592.88;912.87;45;11.7301;148.13;2;2672.1;5;20435.4;0.6692
KARŞIYAKA;AKSOY;A;0.883;0.4838;10756;38.45346;27.10874;678;351;152410;163160;1395.72;1606.18;916.43;14;7.9645;160.59;0;0.0;3;19737.2;0.5323
KARŞIYAKA;ALAYBEY;A;0.801;0.1764;7096;38.45923;27.12348;544;156;75630;82220;1395.75;1606.46;918.08;20;7.9911;160.59;4;1451.6;1;0.0;0.3977
KARŞIYAKA;ATAKENT;A;0.765;0.6985;6498;38.46419;27.08713;202;179;95640;106400;1384.23;1591.79;913.02;1;24.9124;160.59;4;19499.3;4;1071.5;0.3652
KARŞIYAKA;BAHARİYE;A;0.771;0.3407;12088;38.46123;27.11829;629;236;125080;146300;1395.74;1606.49;918.01;29;8.1627;160.59;8;3546.6;6;1295.6;0.5992
KARŞIYAKA;BAHÇELİEVLER;A;0.877;0.8655;25229;38.46592;27.11935;862;562;263030;292560;1395.74;1606.58;917.67;30;9.0863;160.59;12;5368.4;8;2085.8;0.621
KARŞIYAKA;BAHRİYE ÜÇOK;A;0.875;0.4323;12057;38.46094;27.11134;464;337;156480;163270;1395.75;1606.52;917.99;12;7.9783;160.59;3;2233.7;2;171.2;0.5758
KARŞIYAKA;BOSTANLI;A;0.751;1.5091;29995;38.45814;27.09837;1360;1073;422640;475910;1384.24;1591.67;913.19;18;8.7548;160.59;9;4377.0;6;2251.0;0.6894
KARŞIYAKA;CUMHURİYET;B;0.531;1.0862;13425;38.48161;27.09599;2195;1647;224110;340910;1383.25;1586.88;912.16;32;10.6078;160.59;8;1520.1;13;1593.1;0.6177
KARŞIYAKA;DEDEBAŞI;A;0.799;0.5082;20137;38.46917;27.10563;785;303;200290;221890;1395.74;1606.66;917.76;45;8.8787;160.59;5;3322.1;4;588.1;0.6365
KARŞIYAKA;DEMİRKÖPRÜ;A;0.784;0.197;6858;38.46853;27.09966;286;88;72800;83920;1395.74;1606.66;917.76;15;8.8473;160.59;7;3464.4;2;685.6;0.3902
KARŞIYAKA;DONANMACI;A;0.816;0.4376;9631;38.45397;27.11489;814;359;151410;167320;1395.76;1606.37;918.16;15;8.5413;160.59;11;4806.4;15;1207.2;0.6427
KARŞIYAKA;FİKRİ ALTAY;A;0.82;0.1853;8122;38.47291;27.10172;256;128;73770;85180;1395.73;1606.73;917.7;23;9.2884;160.59;2;1122.5;3;784.3;0.3725
KARŞIYAKA;GONCALAR;A;0.833;0.2379;9144;38.46489;27.10548;445;209;100830;112200;1395.74;1606.59;917.82;13;8.8049;160.59;2;189.5;0;0.0;0.4344
KARŞIYAKA;İMBATLI;B;0.606;0.4672;8140;38.47422;27.10914;587;297;108600;136790;1395.73;1606.75;917.68;9;10.9252;160.59;15;6746.6;14;6061.5;0.5165
KARŞIYAKA;İNÖNÜ;A;0.708;0.7026;8152;38.48734;27.09308;679;428;99450;135100;1384.18;1592.13;912.6;16;26.1417;160.59;2;3843.5;0;0.0;0.446
KARŞIYAKA;LATİFE HANIM;B;0.586;2.7733;2316;38.50369;27.12589;302;324;75470;107010;1360.75;1547.95;908.11;8;11.4835;160.59;1;457.8;0;0.0;0.3756
KARŞIYAKA;MAVİŞEHİR;B;0.545;1.5464;13665;38.47091;27.07372;241;374;153670;243390;1384.65;1592.45;912.53;1;203.2514;160.59;4;16373.1;7;1058.1;0.5404
KARŞIYAKA;MUSTAFA KEMAL;B;0.518;1.7133;10485;38.4965;27.09705;197;224;80660;126140;1382.3;1587.84;908.58;8;31.1421;160.59;6;5766.5;4;1977.1;0.3438
KARŞIYAKA;NERGİZ;A;0.833;0.2044;8103;38.46152;27.10512;220;198;87960;93830;1395.75;1606.56;917.83;16;8.14;160.59;0;0.0;0;0.0;0.4004
KARŞIYAKA;ÖRNEKKÖY;A;0.717;1.3673;24026;38.48255;27.10837;1818;916;274790;341160;1395.65;1606.28;915.76;68;11.2199;160.59;4;1603.4;4;1087.5;0.6365
KARŞIYAKA;SANCAKLI;B;0.619;9.5752;172;38.5253;27.16023;190;189;14080;20610;1326.55;1502.65;889.48;0;14.2924;160.59;0;0.0;0;0.0;0.0902
KARŞIYAKA;ŞEMİKLER;A;0.683;1.147;29319;38.47676;27.09211;1641;695;329640;399100;1384.22;1592.02;912.88;77;9.2553;160.59;10;9017.7;14;2458.5;0.7244
KARŞIYAKA;TERSANE;A;0.702;0.3377;6435;38.46276;27.12879;470;228;99570;106500;1395.74;1606.46;917.41;12;8.5013;160.59;0;0.0;1;0.0;0.4333
KARŞIYAKA;TUNA;A;0.877;0.2191;5641;38.45702;27.12096;546;187;96320;111100;1395.75;1606.42;917.94;15;7.8983;160.59;4;1103.0;4;1791.9;0.5385
KARŞIYAKA;YALI;A;0.711;1.8314;36866;38.47388;27.08461;1456;640;382010;488580;1384.22;1591.97;912.92;29;15.0497;160.59;16;11736.5;14;3292.7;0.6148
KARŞIYAKA;YAMANLAR;C;0.477;17.142;159;38.5387;27.1313;237;344;27230;36950;1361.59;1552.43;907.34;0;9.5;160.59;0;0.0;0;0.0;0.151
KARŞIYAKA;ZÜBEYDE HANIM;A;0.69;5.0673;17342;38.50505;27.10944;581;578;192880;250380;1351.13;1540.89;897.73;52;10.9558;160.59;15;17165.3;16;16409.8;0.6831"""

df = pd.read_csv(io.StringIO(GIRDI), sep=';')
ILCELER = ['ÇİĞLİ', 'KARŞIYAKA']
AD = {'ÇİĞLİ': 'Çiğli', 'KARŞIYAKA': 'Karşıyaka'}
print(len(df), 'mahalle |', {AD[k]: int(v) for k, v in df.groupby('ilce').size().items()})
df.head()

## 1 · Varsayımlar

In [ ]:
M2_PER_KWP = 6.0                        # m² başına ~200 W modül
KULLANILABILIR = {0: 0.55, 30: 0.35}    # düz çatı (sıra aralığı + kenar) / 30° beşik çatı (tek yüz)

KISI_BASI_MESKEN = 1059.0               # kWh/kişi/yıl, İzmir 2021 (MDA s.255)
HANE_BUYUKLUGU = 2.8                    # kişi/hane, İzmir 2021 (BP s.89)
HANE_TUKETIM = KISI_BASI_MESKEN * HANE_BUYUKLUGU
HANE_TUKETIM_ESKI = 3600.0

IHTIYAC_AGIRLIK = {'askida': 0.40, 'su_ters': 0.40, 'ses_ters': 0.20}
AGIRLIK = {                             # (teknik, ihtiyaç, erişim)
    'yalnız teknik':     (1.00, 0.00, 0.00),
    'AHP benzeri':       (0.50, 0.35, 0.15),
    'ihtiyaç ağırlıklı': (0.35, 0.50, 0.15),
    'eşit':              (0.34, 0.33, 0.33),
}
ANA = 'AHP benzeri'
print(f'Hane tüketimi: {HANE_TUKETIM:,.1f} kWh/yıl (eski {HANE_TUKETIM_ESKI:,.0f}) '
      f'-> hane eşdeğeri x{HANE_TUKETIM_ESKI / HANE_TUKETIM:.3f}'.replace(',', '.'))

## 2 · Model 1: çatı alanı bandı

Microsoft ayak izleri bitişik binaları birleştirdiği için alanı büyütüyor, OSM ise eksik çizilmiş binalar yüzünden küçültebiliyor. Bu yüzden tek bir alan değil, iki kaynağın arası bant olarak taşınıyor. Alt sınır küçük olan, üst sınır büyük olan kaynak.

In [ ]:
df['alan_alt'] = df[['osm_m2', 'ms_m2']].min(axis=1)
df['alan_ust'] = df[['osm_m2', 'ms_m2']].max(axis=1)
df['alan_orta'] = np.sqrt(df['alan_alt'].clip(lower=1) * df['alan_ust'].clip(lower=1))

g = df.groupby('ilce')
ozet1 = pd.DataFrame({
    'OSM bina': g['osm_n'].sum(),
    'MS bina': g['ms_n'].sum(),
    'OSM alan (ha)': g['osm_m2'].sum() / 1e4,
    'MS alan (ha)': g['ms_m2'].sum() / 1e4,
    'örtüşme medyanı': g['ortusme'].median(),
    'kademe A/B/C': g['kademe'].apply(lambda s: '/'.join(str(int((s == k).sum())) for k in 'ABC')),
}).reindex(ILCELER)
ozet1['MS/OSM alan'] = ozet1['MS alan (ha)'] / ozet1['OSM alan (ha)']
ozet1.rename(index=AD).round(3)

## 3 · Model 2: üretim bandı

* **Alt sınır:** küçük alan × beşik çatı (%35) × kuzeye bakan yüz
* **Üst sınır:** büyük alan × düz çatı (%55) × optimum yön
* **Bant merkezi:** alt ve üst sınırın geometrik ortalaması; ek varsayım gerektirmez
* **En olası yerleşim:** orta alan × düz çatı. Türkiye'de teras çatı yaygın olduğu için eklendi. Bandın ortası değildir, üst sınıra yakın durur.

In [ ]:
df['kWp_alt'] = df['alan_alt'] * KULLANILABILIR[30] / M2_PER_KWP
df['kWp_ust'] = df['alan_ust'] * KULLANILABILIR[0] / M2_PER_KWP
df['kWp_olasi'] = df['alan_orta'] * KULLANILABILIR[0] / M2_PER_KWP
df['kWh_alt'] = df['kWp_alt'] * df['oz_kuzey']
df['kWh_ust'] = df['kWp_ust'] * df['oz_duz']
df['kWh_gm'] = np.sqrt(df['kWh_alt'].clip(lower=1) * df['kWh_ust'])
df['kWh_olasi'] = df['kWp_olasi'] * df['oz_duz']

t = df.groupby('ilce')[['kWp_alt', 'kWp_ust', 'kWh_alt', 'kWh_gm', 'kWh_olasi', 'kWh_ust']].sum().reindex(ILCELER)
t.loc['TOPLAM'] = t.sum()
ozet2 = pd.DataFrame({
    'MWp alt': t['kWp_alt'] / 1e3, 'MWp üst': t['kWp_ust'] / 1e3,
    'GWh alt': t['kWh_alt'] / 1e6, 'GWh bant merkezi': t['kWh_gm'] / 1e6,
    'GWh en olası': t['kWh_olasi'] / 1e6, 'GWh üst': t['kWh_ust'] / 1e6,
})
ozet2['bant (üst/alt)'] = ozet2['GWh üst'] / ozet2['GWh alt']
ozet2.rename(index=AD).round(2)

### Canlı PVGIS kontrolü (isteğe bağlı)

Gömülü özgül verim değerleri PVGIS'ten alındı. `PVGIS_KONTROL = True` yapıp çalıştırırsan bir mahalle için API yeniden sorgulanır ve değerler karşılaştırılır. İnternet gerekir.

In [ ]:
PVGIS_KONTROL = False

if PVGIS_KONTROL:
    import requests
    r0 = df.iloc[0]
    q = requests.get('https://re.jrc.ec.europa.eu/api/v5_3/PVcalc', timeout=90,
                     params=dict(lat=round(r0.lat, 3), lon=round(r0.lon, 3), peakpower=1, loss=14,
                                 angle=0, aspect=0, outputformat='json'))
    canli = q.json()['outputs']['totals']['fixed']['E_y']
    print(f'{r0.mahalle}: gömülü {r0.oz_duz:.1f} | canlı {canli:.1f} kWh/kWp')
else:
    print('Atlandı (PVGIS_KONTROL = False).')

## 4 · Hane eşdeğeri: düzeltmenin etkisi

"Hane eşdeğeri" yıllık üretimin kaç hanenin yıllık mesken elektriğine denk geldiğini gösterir. Tahmini hane sayısı nüfus / 2,8 ile hesaplandı. **Karşılama oranı** yıllık enerji dengesidir: üretimin gündüz yapılması, şebeke ve depolama bu orana dahil değildir.

In [ ]:
h = df.groupby('ilce')[['kWh_alt', 'kWh_gm', 'kWh_ust', 'nufus']].sum().reindex(ILCELER)
h.loc['TOPLAM'] = h.sum()
hane = pd.DataFrame({
    'eski (3.600) merkez': h['kWh_gm'] / HANE_TUKETIM_ESKI,
    'yeni alt': h['kWh_alt'] / HANE_TUKETIM,
    'yeni merkez': h['kWh_gm'] / HANE_TUKETIM,
    'yeni üst': h['kWh_ust'] / HANE_TUKETIM,
    'tahmini hane': h['nufus'] / HANE_BUYUKLUGU,
})
hane['karşılama alt %'] = 100 * hane['yeni alt'] / hane['tahmini hane']
hane['karşılama merkez %'] = 100 * hane['yeni merkez'] / hane['tahmini hane']
print(f"Merkez tahminde hane eşdeğeri artışı: %{100 * (HANE_TUKETIM_ESKI / HANE_TUKETIM - 1):.1f}")
hane.rename(index=AD).round(0).astype(int)

## 5 · Model 3: öncelik skoru

Üç eksen ayrı tutuluyor:

* **Teknik:** yıllık üretimin bant merkezi
* **İhtiyaç:** askıda fatura / 1000 kişi (%40) + düşük su tüketimi (%40) + düşük SES (%20)
* **Erişim:** kamu çatısı m² / 1000 kişi

Erişim ayrı eksende çünkü Çiğli'de kamu binası yoğunluğu ihtiyaçla ters ilişkili. Aynı terimde toplanırsa skor yatırımı ihtiyaçtan uzaklaştırıyor.

Her gösterge 0–1 arasına **sıra tabanlı** yerleştiriliyor (min-max değil; tek bir dev mahalle diğerlerini sıkıştırmasın diye). Aşağıda eski ve düzeltilmiş normalizasyon birlikte hesaplanıyor.

In [ ]:
n = df['nufus'].clip(lower=1)
df['askida_bin'] = df['askida_n'] / n * 1000
df['kamu_bin'] = df['kamu_n'] / n * 1000
df['kamu_cati_bin'] = df['kamu_m2'] / n * 1000
df['cati_kisi'] = df['alan_orta'] / n


def yuzdelik_eski(x):
    """Önceki sürüm: eşit değerler dizideki sıraya göre farklı puan alıyordu."""
    x = np.asarray(x, float)
    s = np.empty(len(x))
    s[np.argsort(x)] = np.arange(len(x))
    return s / (len(x) - 1)


def yuzdelik(x):
    """Düzeltilmiş: eşit değerler ortalama sırayı alır."""
    return (pd.Series(np.asarray(x, float)).rank(method='average').to_numpy() - 1) / (len(x) - 1)


def eksenler(norm):
    a = norm(df['askida_bin'])
    s = norm(-df['su_kisi'])
    ses = norm(-df['ses'])
    return dict(
        teknik=norm(df['kWh_gm']),
        ihtiyac=IHTIYAC_AGIRLIK['askida'] * a + IHTIYAC_AGIRLIK['su_ters'] * s + IHTIYAC_AGIRLIK['ses_ters'] * ses,
        ihtiyac_sessiz=0.5 * a + 0.5 * s,
        erisim=norm(df['kamu_cati_bin']),
        erisim_sayim=norm(df['kamu_bin']),
        ses=ses,
    )


def skor(E, agirlik, ihtiyac='ihtiyac', erisim='erisim'):
    wt, wi, we = agirlik
    return wt * E['teknik'] + wi * E[ihtiyac] + we * E[erisim]


E_eski = eksenler(yuzdelik_eski)
E = eksenler(yuzdelik)

# Doğrulama: eski yöntem yerel modelin önceki çıktısını üretmeli
fark = np.abs(skor(E_eski, AGIRLIK[ANA]) - df['ref_skor_ahp']).max()
print(f'Eski yöntem ile yerel çıktı arasındaki en büyük skor farkı: {fark:.5f}')
assert fark < 1e-3, 'Defter yerel modeli üretemedi'
print('Doğrulama geçti: defter yerel modelle aynı sonucu veriyor.')

### 5.1 · Eşit değer sorunu

SES ilçe düzeyinde yayımlandığı için bir ilçedeki bütün mahallelerde aynı. Birçok mahallede de kamu çatısı yok. Eski normalizasyon bu eşit değerlere farklı puanlar veriyordu.

In [ ]:
for il in ILCELER:
    m = (df['ilce'] == il).to_numpy()
    print(f"{AD[il]:10s} SES {df.loc[m, 'ses'].iloc[0]:.2f} (bütün mahallelerde aynı) | "
          f"düşük SES puanı eski {E_eski['ses'][m].min():.2f}–{E_eski['ses'][m].max():.2f}, "
          f"yeni {E['ses'][m].min():.2f}")

sifir = (df['kamu_m2'] == 0).to_numpy()
print(f"\nKamu çatısı olmayan {sifir.sum()} mahalle | erişim puanı eski "
      f"{E_eski['erisim'][sifir].min():.2f}–{E_eski['erisim'][sifir].max():.2f}, "
      f"yeni {E['erisim'][sifir].min():.2f}")
pd.DataFrame({'ilçe': df['ilce'].map(AD), 'mahalle': df['mahalle'],
              'erişim eski': E_eski['erisim'], 'erişim yeni': E['erisim']})[sifir] \
    .sort_values('erişim eski', ascending=False).round(3).head(10)

### 5.2 · Eksenler arası ilişki (Spearman)

In [ ]:
def rho(a, b):
    """Spearman: ortalama sıralar arasındaki Pearson korelasyonu (eşit değerlere dayanıklı)."""
    ra = pd.Series(np.asarray(a, float)).rank(method='average')
    rb = pd.Series(np.asarray(b, float)).rank(method='average')
    return float(np.corrcoef(ra, rb)[0, 1])

satir = []
for il in ILCELER + [None]:
    m = np.ones(len(df), bool) if il is None else (df['ilce'] == il).to_numpy()
    satir.append({
        'kapsam': 'iki ilçe' if il is None else AD[il], 'n': int(m.sum()),
        'askıda ↔ kamu sayısı': rho(df['askida_bin'][m], df['kamu_bin'][m]),
        'ihtiyaç ↔ erişim': rho(E['ihtiyac'][m], E['erisim'][m]),
        'ihtiyaç ↔ teknik': rho(E['ihtiyac'][m], E['teknik'][m]),
        'su/kişi ↔ çatı/kişi': rho(df['su_kisi'][m], df['cati_kisi'][m]),
    })
pd.DataFrame(satir).set_index('kapsam').round(2)

### 5.3 · Ağırlık senaryoları ve duyarlılık

Kendall tau, bir senaryonun ilk 10'unun ana senaryodaki (AHP benzeri) sırayla ne kadar uyuştuğunu ölçer. 1'e yakın değer, ağırlığın sıralamayı değiştirmediği anlamına gelir.

In [ ]:
ETIKET = (df['mahalle'] + '|' + df['ilce']).to_numpy()


def sirala(s):
    return list(ETIKET[np.argsort(-np.asarray(s), kind='stable')])


def kendall(a, b):
    idx = {v: i for i, v in enumerate(b)}
    o = [x for x in a if x in idx]
    uy = ters = 0
    for i in range(len(o)):
        for j in range(i + 1, len(o)):
            d = idx[o[i]] - idx[o[j]]
            uy += d < 0
            ters += d > 0
    return (uy - ters) / max(uy + ters, 1)


ana_sira = sirala(skor(E, AGIRLIK[ANA]))
tablo = []
for ad, w in AGIRLIK.items():
    s = sirala(skor(E, w))
    tablo.append({'senaryo': ad, 'ilk 10 ortak': len(set(s[:10]) & set(ana_sira[:10])),
                  'Kendall tau': kendall(s[:10], ana_sira)})
for ad, s in [('SES çıkarılırsa', skor(E, AGIRLIK[ANA], ihtiyac='ihtiyac_sessiz')),
              ('kamu alanı yerine sayısı', skor(E, AGIRLIK[ANA], erisim='erisim_sayim'))]:
    s = sirala(s)
    tablo.append({'senaryo': ad, 'ilk 10 ortak': len(set(s[:10]) & set(ana_sira[:10])),
                  'Kendall tau': kendall(s[:10], ana_sira)})
eski_sira = sirala(skor(E_eski, AGIRLIK[ANA]))
tablo.append({'senaryo': 'ESKİ normalizasyon', 'ilk 10 ortak': len(set(eski_sira[:10]) & set(ana_sira[:10])),
              'Kendall tau': kendall(eski_sira[:10], ana_sira)})
pd.DataFrame(tablo).set_index('senaryo').round(2)

### 5.4 · Sonuç: düzeltilmiş sıralamada ilk 15 mahalle

In [ ]:
for ad, w in AGIRLIK.items():
    df['skor_' + ad] = skor(E, w)
df['p_teknik'], df['p_ihtiyac'], df['p_erisim'] = E['teknik'], E['ihtiyac'], E['erisim']
yeni_no = {k: i + 1 for i, k in enumerate(ana_sira)}
eski_no = {k: i + 1 for i, k in enumerate(eski_sira)}
df['sira'] = [yeni_no[k] for k in ETIKET]
df['eski_sira'] = [eski_no[k] for k in ETIKET]

sonuc = df.sort_values('sira')
goster = sonuc[['sira', 'eski_sira', 'ilce', 'mahalle', 'kademe', 'skor_AHP benzeri',
                'p_teknik', 'p_ihtiyac', 'p_erisim', 'kWh_gm', 'nufus']].copy()
goster['ilce'] = goster['ilce'].map(AD)
goster['kWh_gm'] = goster['kWh_gm'] / 1e3
goster = goster.rename(columns={'kWh_gm': 'MWh (merkez)', 'skor_AHP benzeri': 'skor'})
ilk = sonuc.head(15)
print('İlk 15 içinde ilçe dağılımı:', {AD[k]: int(v) for k, v in ilk['ilce'].value_counts().items()},
      '| kademe:', ilk['kademe'].value_counts().sort_index().to_dict())
goster.head(15).round(3).set_index('sira')

In [ ]:
degisen = goster.assign(fark=goster['eski_sira'] - goster['sira'])
degisen = degisen[degisen['fark'].abs() >= 4].sort_values('fark')
print(f"Sırası 4 ve daha fazla değişen mahalle: {len(degisen)}")
degisen[['sira', 'eski_sira', 'fark', 'ilce', 'mahalle', 'p_erisim', 'p_ihtiyac']].round(3).set_index('mahalle')

## 6 · Grafikler

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 2.6))
for i, il in enumerate(ILCELER):
    r = ozet2.loc[il]
    ax.plot([r['GWh alt'], r['GWh üst']], [i, i], color='#0E6B67', lw=3, solid_capstyle='round')
    ax.scatter([r['GWh alt'], r['GWh üst']], [i, i], color='#0E6B67', s=45, zorder=3)
    ax.scatter(r['GWh bant merkezi'], i, marker='|', s=400, color='#1E2A30', zorder=4, linewidths=2)
    ax.annotate(f"{r['GWh alt']:.0f}", (r['GWh alt'], i), xytext=(-8, 0), textcoords='offset points', ha='right', va='center')
    ax.annotate(f"{r['GWh üst']:.0f}", (r['GWh üst'], i), xytext=(8, 0), textcoords='offset points', ha='left', va='center')
    ax.annotate(f"merkez {r['GWh bant merkezi']:.0f}", (r['GWh bant merkezi'], i), xytext=(0, 11),
                textcoords='offset points', ha='center', fontsize=9, color='#3F4D55')
ax.set_yticks(range(len(ILCELER)), [AD[k] for k in ILCELER])
ax.set_ylim(-0.7, len(ILCELER) - 0.3)
ax.set_xlim(0, ozet2.loc[ILCELER, 'GWh üst'].max() * 1.15)
ax.set_xlabel('Yıllık üretim (GWh/yıl)')
ax.set_title('Çatı GES teknik üretim bandı', loc='left')
ax.grid(axis='x', color='#E7ECEE'); ax.set_axisbelow(True)
for k in ['top', 'right', 'left']: ax.spines[k].set_visible(False)
plt.tight_layout(); plt.show()

In [ ]:
en = sonuc.head(12).iloc[::-1]
etiket = [f"{m} ({AD[i]})" for m, i in zip(en['mahalle'], en['ilce'])]
wt, wi, we = AGIRLIK[ANA]
fig, ax = plt.subplots(figsize=(8.5, 5))
sol = np.zeros(len(en))
for ad, w, col, renk in [('teknik', wt, 'p_teknik', '#2F6DB5'), ('ihtiyaç', wi, 'p_ihtiyac', '#D9822B'),
                         ('erişim', we, 'p_erisim', '#3A9E7A')]:
    v = w * en[col].to_numpy()
    ax.barh(etiket, v, left=sol, color=renk, edgecolor='white', linewidth=1.5, label=f'{ad} × {w:.2f}')
    sol += v
for y, s in enumerate(sol):
    ax.text(s + 0.01, y, f'{s:.3f}', va='center', fontsize=9)
ax.set_xlim(0, 1)
ax.set_xlabel('Skor katkısı')
ax.set_title('İlk 12 mahalle: skorun eksenlere dağılımı (AHP benzeri)', loc='left')
ax.legend(loc='lower right', frameon=False)
for k in ['top', 'right']: ax.spines[k].set_visible(False)
plt.tight_layout(); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 5.2))
for il, renk in [('ÇİĞLİ', '#0E6B67'), ('KARŞIYAKA', '#A8650F')]:
    m = df['ilce'] == il
    ax.scatter(df.loc[m, 'eski_sira'], df.loc[m, 'sira'], s=28, color=renk, label=AD[il], alpha=.85)
ax.plot([1, len(df)], [1, len(df)], color='#9AA6AD', lw=1, ls='--')
for _, r in degisen.iterrows():
    ax.annotate(r['mahalle'], (r['eski_sira'], r['sira']), xytext=(4, -3), textcoords='offset points', fontsize=7.5)
ax.set_xlabel('Eski sıra'); ax.set_ylabel('Düzeltilmiş sıra')
ax.set_title('Normalizasyon düzeltmesi sıralamayı nasıl değiştirdi?', loc='left', fontsize=11)
ax.invert_xaxis(); ax.invert_yaxis()
ax.legend(frameon=False)
plt.tight_layout(); plt.show()

## 7 · Çıktı dosyası

Sonuç tablosu çalışma klasörüne yazılır. Colab'da soldaki **Dosyalar** panelinden indirilebilir. `INDIR = True` yaparsan tarayıcı indirmesi de başlar.

In [ ]:
INDIR = False

sutun = ['sira', 'eski_sira', 'ilce', 'mahalle', 'kademe', 'ortusme', 'nufus',
         'alan_alt', 'alan_ust', 'kWp_alt', 'kWp_ust', 'kWh_alt', 'kWh_gm', 'kWh_olasi', 'kWh_ust',
         'askida_bin', 'su_kisi', 'ses', 'kamu_n', 'kamu_m2', 'kamu_cati_bin',
         'p_teknik', 'p_ihtiyac', 'p_erisim'] + ['skor_' + a for a in AGIRLIK]
dosya = 'catigunes_model_sonuclari.csv'
sonuc[sutun].round(4).to_csv(dosya, sep=';', index=False, encoding='utf-8-sig')
print('Yazıldı:', dosya, '|', len(sonuc), 'satır')
if INDIR:
    from google.colab import files
    files.download(dosya)

## 8 · Sınırlılıklar

* Çatı alanı SAM segmentasyonundan değil bina ayak izinden geliyor; çatı eğimi ve yönü mahalle ölçeğinde doğrulanmadı.
* Hane tüketimi ve hane büyüklüğü **il** ortalaması. Çiğli ve Karşıyaka için ilçe verisi (TÜİK ADNKS hane büyüklüğü, GDZ mahalle tüketimi) istenmeli.
* Karşılama oranı yıllık enerji dengesidir; saatlik eşleşme, şebeke ve depolama kısıtı içermez.
* SES ilçe düzeyinde; mahalleler arasında ayrım üretmez, yalnızca iki ilçeyi ayırır.
* Askıda fatura verisi 2020 pandemi programından; seçilim yanlılığı taşır, tek başına yoksulluk ölçüsü değildir.
* Plan belgelerinin işaret ettiği risk katmanları (sel ve kıyı riski, Kaklıç havaalanı, Sasalı doğal sit, bina kalitesi) henüz modelde yok.